# ROA analysis — what moved it, and by how much

Reads the **ROA formula** out of both workbooks, follows it down to the inputs that actually changed,
names them from column B, and writes an interactive page where you tick any combination of components
and the real formula is re-evaluated.

Needs these files in the same folder as this notebook:

| File | Role |
|---|---|
| `formula_trace.py` | Parses the formula, walks its precedents, finds what changed |
| `roa_explorer.py` | Renders the interactive page |
| `vintage_compare.py` | Supplies the row map (the inserted row after 200) |

---

## Before you start: the workbooks must be real `.xlsx`

**Renaming `Report.xlsb` to `Report.xlsx` does not work.** They are different file formats, not
different labels — a renamed `.xlsb` will fail to open or open as nonsense. Convert properly:

> **Excel → File → Save As → Excel Workbook (\*.xlsx)**

This is only needed for the *formula* trace. `pyxlsb` and `xlrd` expose cached values and nothing
else — the formula text simply is not available through them — so a `.xlsb` cannot be traced at all.
The value comparison in `vintage_delta_comparison.ipynb` still works on the original `.xlsb`.

`load_book()` refuses a `.xlsb` with that instruction rather than failing obscurely.

While you are in Excel, **let it recalculate and save**. That stores the cached values, which this
notebook cross-checks its own arithmetic against.

---

## Why ticking components beats a waterfall

ROA is net income ÷ assets — a **ratio** — so its drivers do not add up. On the worked example the
components' individual effects sum to −12.196 bps while the true combined move is −12.055 bps. That
0.141 bps gap is real interaction, and a static waterfall has to either hide it or allocate it
arbitrarily. Re-evaluating the actual formula for whatever subset you select is the only way to answer
*"what did these two changes do together"* correctly.

## What counts as a component

A cell becomes a component when **it changed and nothing labelled below it changed**. So `Net income`
and `Total revenue` are walked *through* rather than reported, and what you get back is the individual
line — `Credit provision`, `Interest income` — under its column B name.

## The integrity check

`Trace.check()` re-evaluates the rebuilt formula with all-old and then all-new inputs and asserts both
reproduce the workbook's own cached values. If the formula was followed incorrectly it **raises**
instead of returning a plausible wrong number. Section 4 runs it on every vintage.

## 1. Imports

In [ ]:
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import formula_trace as ft
import roa_explorer as rx
import vintage_compare as vc
for m in (ft, rx, vc):
    importlib.reload(m)

try:
    import pandas as pd
except ImportError:
    pd = None

print("loaded from", Path(ft.__file__).parent)
print("formula support:", ", ".join(sorted(set(ft.FUNCTIONS) | ft.LAZY_FUNCTIONS)))

## 2. Settings
`ROA_ROW_OLD` is the row in the **old** workbook. The new one is worked out from the row map, so you
do not enter 210 anywhere — if the map is right, 209 lands on 210 by itself. Section 3 shows you that
it did.

In [ ]:
# ============================================================================
#  EDIT THIS CELL
# ============================================================================

OLD_FILE = "vintage_report.xlsx"        # the current process (no "snow" in the name)
NEW_FILE = "vintage_report_snow.xlsx"   # the snow process - one row longer, and the new ROA
OUTPUT_PAGE = "roa_explorer.html"       # the interactive page written here

N_VINTAGES   = 24
VINTAGE_NAME = "Vintage{}"              # matched loosely: "Vintage 1", "VINTAGE_01", ...

ROA_ROW_OLD = 209                       # ROA row in the OLD workbook (210 in snow, via the row map)
ROA_COL     = "C"                       # the column holding the ROA figure - check this in section 3
LABEL_COL   = "B"                       # component names come from here
MAX_DEPTH   = 8                         # how far down the formula chain to follow

# The inserted row: everything past row 200 sits one lower in the snow workbook.
VINTAGE_SPEC = vc.vintage_specs(n_vintages=N_VINTAGES, vintage_name=VINTAGE_NAME)[1]
_ROW_MAP, _ = vc.build_row_map(VINTAGE_SPEC)
def row_map(r: int) -> int:
    return _ROW_MAP.get(r, r if r <= 200 else r + 1)

from openpyxl.utils import column_index_from_string
ROA_COL_IDX, LABEL_COL_IDX = column_index_from_string(ROA_COL), column_index_from_string(LABEL_COL)

print(f"ROA: old {ROA_COL}{ROA_ROW_OLD}  ->  new {ROA_COL}{row_map(ROA_ROW_OLD)}")
print(f"a row before the insertion, for contrast: old 150 -> new {row_map(150)}")
assert row_map(ROA_ROW_OLD) == ROA_ROW_OLD + 1, "expected the ROA row to shift by one"

## 3. Check the ROA cell before tracing anything
Prints what is actually in the ROA row of both workbooks: the label in column B, the formula, the
cached value, and which columns along that row even hold formulas.

**What you want to see:** the same label both sides, a formula in `ROA_COL`, and a cached value. If
the cached value is empty, Excel never saved a calculated result — re-save the file. If the formula
sits in a different column, change `ROA_COL` above.

In [ ]:
books = {}
for tag, path in (("old", OLD_FILE), ("new", NEW_FILE)):
    if not Path(path).exists():
        print(f"{tag}: NOT FOUND - {path}")
        continue
    try:
        books[tag] = ft.load_book(path)
        print(f"{tag}: loaded {Path(path).name}  ({len(books[tag].sheets)} sheets)")
    except ft.UnsupportedFormula as exc:
        print(f"{tag}: {exc}")

if len(books) == 2:
    sheet_old = vc.resolve_sheet(VINTAGE_NAME.format(1), books["old"].sheets)
    sheet_new = vc.resolve_sheet(VINTAGE_NAME.format(1), books["new"].sheets)
    print(f"\nfirst vintage sheet: old {sheet_old!r}, new {sheet_new!r}\n")

    for tag, sheet, row in (("old", sheet_old, ROA_ROW_OLD),
                            ("new", sheet_new, row_map(ROA_ROW_OLD))):
        b = books[tag]
        print(f"{tag}  {sheet}!{ROA_COL}{row}")
        print(f"   column {LABEL_COL} says : {b.value(sheet, row, LABEL_COL_IDX)!r}")
        print(f"   formula          : {b.formula(sheet, row, ROA_COL_IDX)!r}")
        print(f"   cached value     : {b.value(sheet, row, ROA_COL_IDX)!r}")

    b = books["old"]
    cols = [(c, b.formula(sheet_old, ROA_ROW_OLD, c))
            for c in range(1, 28) if b.formula(sheet_old, ROA_ROW_OLD, c)]
    from openpyxl.utils import get_column_letter
    print(f"\ncolumns on row {ROA_ROW_OLD} that hold a formula: "
          f"{', '.join(get_column_letter(c) for c, _ in cols) or 'none'}")
    if cols and ROA_COL_IDX not in [c for c, _ in cols]:
        print(f"   ROA_COL is {ROA_COL}, which holds no formula - pick one of the above.")

## 4. Trace every vintage
Follows the formula on each sheet and checks the result reproduces both cached values. A sheet that
cannot be traced is reported with the reason and skipped, rather than quietly dropping out.

In [ ]:
traces, names, failures = [], [], []

if len(books) == 2:
    for i in range(1, N_VINTAGES + 1):
        want = VINTAGE_NAME.format(i)
        s_old = vc.resolve_sheet(want, books["old"].sheets)
        s_new = vc.resolve_sheet(want, books["new"].sheets)
        if s_old is None or s_new is None:
            failures.append((want, f"sheet not found (old={s_old}, new={s_new})"))
            continue
        try:
            tr = ft.trace(books["old"], books["new"], s_old, ROA_ROW_OLD, ROA_COL_IDX,
                          row_map=row_map, label_col=LABEL_COL_IDX,
                          max_depth=MAX_DEPTH, sheet_new=s_new)
            tr.check()                     # raises if the formula was followed incorrectly
        except (ft.FormulaError, AssertionError) as exc:
            failures.append((want, str(exc)))
            continue
        traces.append(tr)
        names.append(s_old)

    print(f"{len(traces)} of {N_VINTAGES} vintages traced and verified\n")
    if traces:
        print(f"{'sheet':<14}{'old ROA':>10}{'new ROA':>10}{'delta':>11}{'components':>12}")
        for tr in traces:
            d = (tr.value_new - tr.value_old) * 10_000
            print(f"{tr.sheet:<14}{tr.value_old:>10.4%}{tr.value_new:>10.4%}"
                  f"{d:>+10.1f}b{len(tr.components):>12}")
    for name, why in failures:
        print(f"\nSKIPPED {name}: {why}")
else:
    print("Load both workbooks in section 3 first.")

## 5. Write the interactive page
One self-contained HTML file — no assets to send alongside it. Open it, or email it to whoever asked.

In [ ]:
if traces:
    payload = rx.build_payload(traces, names=names)
    page = rx.write_html(
        payload, OUTPUT_PAGE,
        title="What moved ROA, component by component",
        subtitle=f"{Path(OLD_FILE).stem} → {Path(NEW_FILE).stem} · "
                 f"ROA at {ROA_COL}{ROA_ROW_OLD} / {ROA_COL}{row_map(ROA_ROW_OLD)}",
        sample=False)
    print(f"written: {page}  ({page.stat().st_size:,} bytes)")
    print("Tick components on the page; selecting all of them lands exactly on the new ROA.")
else:
    print("Nothing traced - fix section 4 first.")

## 6. The same thing as a table
Every component across every vintage, and the components that show up again and again — those are the
ones worth explaining to whoever signs this off.

In [ ]:
if not traces:
    print("Run section 4 first.")
elif pd is None:
    print("pandas is not installed - `pip install pandas` to use this section.")
else:
    rows = []
    for tr in traces:
        base = tr.evaluate_with(set())
        for c in tr.components:
            rows.append({
                "Sheet": tr.sheet, "Component": c.label,
                "Cell (old)": c.ref_old, "Cell (new)": c.ref_new,
                "Old": c.old, "New": c.new, "Change": c.delta,
                "Solo bps": (tr.evaluate_with({c.key}) - base) * 10_000,
            })
    comp = pd.DataFrame(rows)
    print(f"{len(comp):,} changed components across {comp['Sheet'].nunique()} sheets\n")

    print("Components by total ROA effect")
    display(comp.groupby("Component")
                .agg(sheets=("Sheet", "nunique"), total_change=("Change", "sum"),
                     total_bps=("Solo bps", "sum"), worst_bps=("Solo bps", "min"))
                .sort_values("total_bps"))

    print("\nLargest single effects")
    display(comp.reindex(comp["Solo bps"].abs().sort_values(ascending=False).index).head(20))

    # interaction: the gap between adding the parts up and evaluating them together
    print("\nWhy the parts do not add up")
    inter = []
    for tr in traces:
        base = tr.evaluate_with(set())
        solo = sum(tr.evaluate_with({c.key}) - base for c in tr.components) * 10_000
        both = (tr.evaluate_with({c.key for c in tr.components}) - base) * 10_000
        inter.append({"Sheet": tr.sheet, "Sum of parts (bps)": solo,
                      "All together (bps)": both, "Interaction (bps)": both - solo})
    display(pd.DataFrame(inter).round(3))

## 7. Self-test on generated workbooks
Builds a small pair of workbooks with a real formula chain — revenue and expense lines feeding
subtotals, feeding net income, over average assets — with the snow copy one row longer past row 200,
so ROA genuinely sits at 209 in one and 210 in the other. Then asserts the trace finds the individual
lines rather than the subtotals, and that it reproduces the arithmetic.

Run this first if you want to see the whole thing work before your own files are ready.

In [ ]:
RUN_SELF_TEST = True        # set to False once you are pointing at the real workbooks

if RUN_SELF_TEST:
    import tempfile
    from openpyxl import Workbook

    demo_dir = Path(tempfile.mkdtemp(prefix="roa_selftest_"))
    N_DEMO = 3

    REVENUE = [(150, "Interest income", 214.5), (152, "Fee income", 18.2), (154, "Other income", 3.1)]
    EXPENSE = [(170, "Funding cost", 96.4), (172, "Credit provision", 31.7),
               (174, "Operating expense", 44.25)]
    MOVED = {150: -2.1, 172: +2.8, 174: -0.6, 196: +120.0}     # what the snow process changes

    def build_demo(path, snow):
        wb = Workbook(); wb.remove(wb.active)
        for v in range(1, N_DEMO + 1):
            ws = wb.create_sheet(f"Vintage {v}")
            def put(r, label, value=None, formula=None):
                ws.cell(row=r, column=1, value=f"code{r}")
                ws.cell(row=r, column=2, value=label)
                ws.cell(row=r, column=3, value=formula if formula else value)
            for r, lab, val in REVENUE:
                put(r, lab, val + MOVED.get(r, 0) if snow else val)
            put(165, "Total revenue",   formula="=SUM(C150:C160)")
            for r, lab, val in EXPENSE:
                put(r, lab, val + MOVED.get(r, 0) if snow else val)
            put(185, "Total expenses",  formula="=SUM(C170:C180)")
            put(190, "Net income",      formula="=C165-C185")
            put(195, "Opening assets",  4180.0)
            put(196, "Closing assets",  4320.0 + MOVED.get(196, 0) if snow else 4320.0)
            put(198, "Average assets",  formula="=AVERAGE(C195:C196)")
            put(210 if snow else 209, "Return on assets", formula="=IFERROR(C190/C198,0)")
        wb.save(path)

    d_old, d_new = demo_dir / "demo_old.xlsx", demo_dir / "demo_snow.xlsx"
    build_demo(d_old, False); build_demo(d_new, True)

    b_old, b_new = ft.load_book(d_old), ft.load_book(d_new)
    demo_traces = [ft.trace(b_old, b_new, f"Vintage {v}", 209, 3,
                            row_map=row_map, label_col=2) for v in range(1, N_DEMO + 1)]
    for t in demo_traces:
        t.check()
    tr = demo_traces[0]

    print(f"formula   {tr.sheet}!{tr.ref_old}: {tr.formula_old}")
    print(f"          {tr.sheet}!{tr.ref_new}: {tr.formula_new}    (changed: {tr.formula_changed})")
    print(f"ROA       {tr.value_old:.4%}  ->  {tr.value_new:.4%}   "
          f"{(tr.value_new - tr.value_old) * 10_000:+.2f} bps\n")

    base = tr.evaluate_with(set())
    print(f"{'component':<22}{'cell':>7}{'old':>10}{'new':>10}{'solo bps':>11}")
    solo = {}
    for c in tr.components:
        solo[c.key] = (tr.evaluate_with({c.key}) - base) * 10_000
        print(f"{c.label:<22}{c.ref_old:>7}{c.old:>10,.2f}{c.new:>10,.2f}{solo[c.key]:>+11.2f}")

    found = {c.label for c in tr.components}
    assert found == {"Interest income", "Credit provision", "Operating expense", "Closing assets"}, found
    assert not (found & {"Net income", "Total revenue", "Total expenses", "Average assets"}), \
        "subtotals must be walked through, not reported as what moved"
    assert all(c.labelled for c in tr.components), "every component should carry a column B name"
    assert tr.row_old == 209 and tr.row_new == 210, (tr.row_old, tr.row_new)
    assert all(c.row_old == c.row_new for c in tr.components), \
        "these components sit above row 200, so they do not shift"

    together = (tr.evaluate_with({c.key for c in tr.components}) - base) * 10_000
    assert abs(together - (tr.value_new - tr.value_old) * 10_000) < 1e-6, \
        "selecting every component must land exactly on the new ROA"
    gap = together - sum(solo.values())
    print(f"\nsum of the parts {sum(solo.values()):+.3f} bps   all together {together:+.3f} bps"
          f"   interaction {gap:+.3f} bps")
    assert abs(gap) > 0.01, "a ratio's drivers should not be additive"

    demo_page = rx.write_html(
        rx.build_payload(demo_traces, names=[t.sheet for t in demo_traces]),
        demo_dir / "demo_roa_explorer.html",
        title="What moved ROA, component by component",
        subtitle="Generated demo workbooks · ROA at C209 / C210", sample=True)

    print("\nself-test")
    print("  components are the individual lines, the row map puts 209 against 210,")
    print("  and every subset re-evaluates the real formula")
    print(f"  page: {demo_page}")